# mine-tuning-model

## 필수 라이브러리 설치

In [1]:
!pip uninstall -y transformers trl -q
!pip install transformers==4.51.3 trl==0.17.0 -q
!pip install datasets peft accelerate bitsandbytes -q
!pip install torchao --upgrade -q
!pip install wandb -q

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip freeze > /content/drive/MyDrive/Colab\ Notebooks/Github/mine-tuning-model/requirements-ft.txt

In [4]:
!wandb login

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: ddorin789 (ddorin789-nothing) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# 데이터 로드 및 Instruction 형식 변환

In [5]:
from datasets import load_dataset

# 페르소나 데이터 로드
data_path = "/content/drive/MyDrive/Colab Notebooks/Github/mine-tuning-model/data/persona.jsonl"
ds = load_dataset("json", data_files=data_path)
print(ds)

# Instruction 형식으로 변환
def format_instruction(example):
    return {
        "text": f"""<|im_start|>system
You are a knowledgeable Minecraft expert assistant. Your tone is polite but approachable.
<|im_end|>
<|im_start|>user
{example['question']}
<|im_end|>
<|im_start|>assistant
{example['answer']}
<|im_end|>"""
    }

train_data = ds["train"].map(format_instruction, remove_columns=["question", "answer"])
print(f"✅ 데이터 변환 완료: {len(train_data)}개")
print("\n샘플 확인:")
print(train_data[0]["text"])

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 50
    })
})


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

✅ 데이터 변환 완료: 50개

샘플 확인:
<|im_start|>system
You are a knowledgeable Minecraft expert assistant. Your tone is polite but approachable.
<|im_end|>
<|im_start|>user
Where can I find diamonds in Minecraft?
<|im_end|>
<|im_start|>assistant
Diamonds are most commonly found around Y-level -58. Exploring caves or using branch mining can help you gather them more efficiently.
<|im_end|>


# 모델, 토크나이저 로드

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

model_id = "Qwen/Qwen3-8B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

print(f"✅ 모델 로드 완료!")
print(f"GPU 사용: {torch.cuda.memory_allocated()/1024**3:.1f}GB / 22.5GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✅ 모델 로드 완료!
GPU 사용: 5.7GB / 22.5GB


# LoRa 어댑터 설정

In [7]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 4bit 학습 준비 추가
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj",
        "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 43,646,976 || all params: 8,234,382,336 || trainable%: 0.5301


# 학습 실행(SFTTrainer)

In [ ]:
from trl import SFTTrainer, SFTConfig

DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks/Github/mine-tuning-model"

training_args = SFTConfig(
    output_dir=f"{DRIVE_DIR}/persona-lora",
    num_train_epochs=8,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    warmup_steps=20,
    lr_scheduler_type="cosine",
    report_to="wandb",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    args=training_args,
    processing_class=tokenizer,
)

print("🚀 페르소나 파인튜닝 시작!")
trainer.train()

save_dir = f"{DRIVE_DIR}/persona-lora"
trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"✅ 저장 완료! → {save_dir}")

wandb.finish()

Converting train dataset to ChatML:   0%|          | 0/50 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


🚀 페르소나 파인튜닝 시작!


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: ddorin789 (ddorin789-nothing) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss


In [ ]:
import transformers
import trl

print(transformers.__version__)
print(trl.__version__)

5.0.0
1.4.0
